<a href="https://colab.research.google.com/github/umslengineering/EE1108/blob/main/EE1108_YOLO_builtin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Laptop Built-in Camera for YOLOv8 Object Detection

In [ ]:
"""
Laptop Webcam Viewer + YOLOv8 Object Detection
-----------------------------------------------
Requirements:
    pip install opencv-python numpy ultralytics

On first run, ultralytics will auto-download yolov8n.pt (~6 MB).
"""

# Import OpenCV for camera access, image processing, and display
import cv2
# Import NumPy for numerical operations and array manipulation
import numpy as np
# Import time for FPS calculation
import time
# Import YOLO from ultralytics for object detection
from ultralytics import YOLO

# ── Config ────────────────────────────────────────────────────────────────────
# Camera index — 0 = built-in webcam, 1 = first external camera, etc.
CAMERA_INDEX = 0

# YOLO model size: n=nano (fastest/least accurate) up to x=extra-large (slowest/most accurate)
MODEL_NAME  = "yolov8x.pt"

# Minimum confidence score (0.0–1.0) a detection must reach to be displayed
CONF_THRESH = 0.45

# Whitelist of class names to show (e.g. ["person", "cat"]), or None to show all classes
FILTER_CLASSES = None

# How often (in frames) to run the YOLO model — 1 = every frame, 2 = every other frame, etc.
DETECT_EVERY = 1
# ─────────────────────────────────────────────────────────────────────────────


# Generates a unique, consistent BGR colour for each YOLO class ID
def class_colour(class_id: int) -> tuple:
    # Seed NumPy's RNG with the class ID so the same class always gets the same colour
    np.random.seed(class_id + 42)
    # Return a tuple of three random integers in [80, 255) as a BGR colour
    return tuple(int(c) for c in np.random.randint(80, 255, 3))


def open_camera(index: int) -> cv2.VideoCapture:
    """Opens the webcam at the given index and returns the VideoCapture object."""
    # Create a VideoCapture object targeting the chosen camera index
    cap = cv2.VideoCapture(index)
    # Raise an error immediately if the camera could not be opened
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open camera at index {index}. "
                           "Try a different CAMERA_INDEX value.")
    # Set the capture resolution to 1280x720 (falls back to camera's default if unsupported)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    print(f"✅ Camera opened  –  "
          f"{int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))}x"
          f"{int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))} @ "
          f"{int(cap.get(cv2.CAP_PROP_FPS))} FPS")
    return cap


def read_camera_frame(cap: cv2.VideoCapture):
    """
    Reads a single frame from the VideoCapture object.
    Returns the frame, or None if the read failed.
    """
    # Attempt to grab and decode the next frame from the camera
    ret, frame = cap.read()
    # Return None to signal a failed read (e.g. camera disconnected)
    if not ret:
        return None
    return frame


def draw_detections(frame: np.ndarray, results, names: dict,
                    filter_classes=None, conf_thresh=CONF_THRESH) -> int:
    """
    Draw bounding boxes + labels on frame in-place.
    Returns the number of objects drawn.
    """
    # Counter for how many detections are actually rendered
    count = 0
    # Extract the Boxes object from the first (and only) result
    boxes = results[0].boxes

    # Loop over every detected bounding box
    for box in boxes:
        # Scalar confidence score for this detection
        conf     = float(box.conf[0])
        # Integer class index for this detection
        class_id = int(box.cls[0])
        # Human-readable class name looked up from the model's name dictionary
        label    = names[class_id]

        # Skip this detection if a class whitelist is active and the label isn't in it
        if filter_classes and label not in filter_classes:
            continue
        # Skip this detection if its confidence is below the current threshold
        if conf < conf_thresh:
            continue

        # Unpack the bounding box corners from the tensor and convert to plain ints
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        # Pick a consistent BGR colour for this class
        colour = class_colour(class_id)

        # Draw the bounding box rectangle on the frame (thickness = 2 px)
        cv2.rectangle(frame, (x1, y1), (x2, y2), colour, 2)

        # Build the label string showing the class name and confidence as a percentage
        text = f"{label}  {conf:.0%}"
        # Measure the pixel dimensions of the rendered label text
        (tw, th), baseline = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
        # Draw a filled rectangle behind the label text for readability
        cv2.rectangle(frame, (x1, y1 - th - baseline - 6), (x1 + tw + 4, y1), colour, -1)

        # Draw the label text in black over the coloured background rectangle
        cv2.putText(
            frame, text,
            (x1 + 2, y1 - baseline - 2),       # Slightly inset from the background rectangle
            cv2.FONT_HERSHEY_SIMPLEX, 0.55,
            (0, 0, 0), 1, cv2.LINE_AA,           # Black text with anti-aliasing
        )
        # Increment the counter for each successfully drawn detection
        count += 1

    return count


def draw_hud(frame: np.ndarray, fps: float, obj_count: int,
             detection_on: bool, model_name: str) -> None:
    """Draws the HUD bar at the top of the frame."""
    # Get frame height and width for positioning HUD elements
    h, w = frame.shape[:2]

    # Copy the frame to use as an overlay for the semi-transparent HUD background
    overlay = frame.copy()
    # Draw a solid dark rectangle across the full width, 44 px tall
    cv2.rectangle(overlay, (0, 0), (w, 44), (20, 20, 20), -1)
    # Blend the dark overlay (60%) with the original frame (40%) for a transparency effect
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

    # Choose green when detection is active, blue/grey when it is off
    status_col = (0, 220, 80) if detection_on else (60, 60, 200)
    # Text label that reflects the current detection state
    status_txt = "DETECT ON" if detection_on else "DETECT OFF"

    # Render the current FPS value on the left side of the HUD bar
    cv2.putText(frame, f"FPS: {fps:5.1f}", (8, 28),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 220, 80), 2, cv2.LINE_AA)

    # Render the object count to the right of the FPS label
    cv2.putText(frame, f"Objects: {obj_count}", (140, 28),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 180, 255), 2, cv2.LINE_AA)

    # Render the detection on/off status label on the right side of the HUD
    cv2.putText(frame, status_txt, (w - 170, 28),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, status_col, 2, cv2.LINE_AA)

    # Render the active model filename centred in the HUD bar
    cv2.putText(frame, model_name, (w // 2 - 50, 28),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (180, 180, 180), 1, cv2.LINE_AA)


def print_help():
    # Print the keyboard shortcut reference box to the terminal on startup
    print("""
╔══════════════════════════════════════════╗
║     Webcam  Object Detection Viewer     ║
╠══════════════════════════════════════════╣
║  D  – toggle detection on/off           ║
║  S  – save current frame as PNG         ║
║  +  – increase confidence threshold     ║
║  -  – decrease confidence threshold     ║
║  M  – mirror / flip frame horizontally  ║
║  Q / ESC – quit                         ║
╚══════════════════════════════════════════╝
""")


def main():
    # Show keyboard controls in the terminal before anything else runs
    print_help()

    # ── Load model ────────────────────────────────────────────────────────────
    print(f"Loading model: {MODEL_NAME} ...")
    # Load (or download) the specified YOLO model weights from disk
    model = YOLO(MODEL_NAME)
    # Retrieve the dictionary mapping class integer IDs to human-readable names
    names = model.names
    print(f"✅ Model ready  –  {len(names)} classes")

    # ── Open camera ───────────────────────────────────────────────────────────
    print(f"Opening camera index {CAMERA_INDEX} ...")
    # Attempt to open the built-in webcam; raises RuntimeError if unavailable
    cap = open_camera(CAMERA_INDEX)

    # ── Runtime state ─────────────────────────────────────────────────────────
    # Flag controlling whether YOLO inference is currently active
    detection_on  = True
    # Local copy of the confidence threshold that can be adjusted at runtime
    conf_thresh   = CONF_THRESH
    # Flag controlling whether the frame is flipped horizontally (mirror mode)
    mirror        = True          # Default ON — feels more natural on a webcam
    # Timestamp used as the start of the current FPS measurement window
    fps_start     = time.time()
    # Number of frames received within the current FPS window
    frame_count   = 0
    # Most recently computed FPS value shown on the HUD
    fps_display   = 0.0
    # Number of objects drawn in the most recent detection pass
    obj_count     = 0
    # Cached YOLO results from the last time inference was run
    last_results  = None
    # Absolute frame counter used to implement the DETECT_EVERY skip logic
    frame_idx     = 0
    # Running counter used to generate unique filenames for saved screenshots
    save_counter  = 0

    # ── Main loop ─────────────────────────────────────────────────────────────
    try:
        while True:
            # Read one frame from the webcam
            frame = read_camera_frame(cap)

            # If the camera returns no frame, it has likely been disconnected
            if frame is None:
                print("Camera read failed — check that the webcam is connected.")
                break

            # Flip the frame horizontally so it behaves like a mirror (optional)
            if mirror:
                frame = cv2.flip(frame, 1)

            # Increment the FPS window counter for every received frame
            frame_count += 1
            # Increment the absolute frame index used for DETECT_EVERY logic
            frame_idx   += 1
            # Capture the current wall-clock time for FPS calculation
            now     = time.time()
            # Compute how many seconds have elapsed in this FPS window
            elapsed = now - fps_start

            # Recalculate displayed FPS once per second
            if elapsed >= 1.0:
                fps_display = frame_count / elapsed
                # Reset the window start time and frame counter for the next window
                fps_start   = now
                frame_count = 0

            # ── Run detection ─────────────────────────────────────────────
            # Only run inference if detection is enabled AND on a scheduled interval
            if detection_on and frame_idx % DETECT_EVERY == 0:
                last_results = model(
                    frame,            # Pass the BGR frame directly to YOLO
                    conf=conf_thresh, # Apply the current runtime confidence threshold
                    verbose=False,    # Suppress per-inference console output
                    stream=False,     # Return a list of Results, not a generator
                )

            # ── Draw detections ───────────────────────────────────────────
            # Only draw if detection is active and we have at least one cached result
            if detection_on and last_results is not None:
                obj_count = draw_detections(
                    frame, last_results, names, FILTER_CLASSES, conf_thresh
                )
            elif not detection_on:
                # Reset the object count immediately when detection is toggled off
                obj_count = 0

            # ── HUD ───────────────────────────────────────────────────────
            # Overlay FPS, object count, detection status, and model name on the frame
            draw_hud(frame, fps_display, obj_count, detection_on, MODEL_NAME)

            # Display the annotated frame in an OpenCV window
            cv2.imshow("Webcam  |  Object Detection  |  D=toggle  Q=quit", frame)

            # ── Keyboard ──────────────────────────────────────────────────
            # Poll for a keypress, waiting 1 ms so the display loop stays responsive
            key = cv2.waitKey(1) & 0xFF

            # Q key or ESC — cleanly exit the application
            if key in (ord("q"), 27):
                print("Quitting.")
                break

            # D key — toggle YOLO inference on or off
            elif key == ord("d"):
                detection_on = not detection_on
                # Clear stale results so boxes disappear immediately when toggled off
                last_results = None
                obj_count    = 0
                print(f"Detection {'ON' if detection_on else 'OFF'}")

            # S key — write the current annotated frame to a PNG file
            elif key == ord("s"):
                fname = f"capture_{save_counter:04d}.png"
                cv2.imwrite(fname, frame)
                # Advance counter so the next save gets a unique filename
                save_counter += 1
                print(f"Saved {fname}")

            # + key — raise the confidence threshold by 5 percentage points (max 0.95)
            elif key == ord("+"):
                conf_thresh = min(0.95, conf_thresh + 0.05)
                print(f"Confidence threshold: {conf_thresh:.2f}")

            # - key — lower the confidence threshold by 5 percentage points (min 0.05)
            elif key == ord("-"):
                conf_thresh = max(0.05, conf_thresh - 0.05)
                print(f"Confidence threshold: {conf_thresh:.2f}")

            # M key — toggle mirror/flip mode on or off
            elif key == ord("m"):
                mirror = not mirror
                print(f"Mirror {'ON' if mirror else 'OFF'}")

    # Allow Ctrl+C in the terminal to exit the loop gracefully
    except KeyboardInterrupt:
        print("\nStopped by user.")

    # ── Cleanup ───────────────────────────────────────────────────────────────
    finally:
        # Always release the camera so other applications can use it
        cap.release()
        # Close all OpenCV windows regardless of how the loop ended
        cv2.destroyAllWindows()
        print("Done.")


# Only run main() when this file is executed directly, not when imported as a module
if __name__ == "__main__":
    main()